## Downscaling modeling with LightGBM

Name the features in the dataset.

In [ ]:
features = [
    "era5_wbgt",

    # Hourly meteorology
    "solar", "Tair", "relhum", "speed",
    "elev_m", "cza", "fdir", "pres",
    "zspeed", "dT",
    
    "urban",

    # ASTER terrain statistics
    "aster_elev_mean", "aster_elev_sd",
    "aster_elev_min", "aster_elev_max",
    "aster_elev_range", "aster_elev_median",
    "aster_elev_q25", "aster_elev_q75",
    "aster_elev_iqr",

    # NLCD land-cover variables
    "nlcd_urban", "nlcd_imp_frac", "nlcd_tree_frac",
    "nlcd_LC11_frac", "nlcd_LC21_frac",
    "nlcd_LC22_frac", "nlcd_LC23_frac",
    "nlcd_LC24_frac", "nlcd_LC31_frac",
    "nlcd_LC41_frac", "nlcd_LC42_frac",
    "nlcd_LC43_frac", "nlcd_LC52_frac",
    "nlcd_LC71_frac", "nlcd_LC81_frac",
    "nlcd_LC82_frac", "nlcd_LC90_frac",
    "nlcd_LC95_frac", "nlcd_developed_frac",
    "nlcd_forest_frac", "nlcd_shrub_frac",
    "nlcd_water_frac", "nlcd_crop_frac",
    "nlcd_wetland_frac",

    # Vegetation
    "modis_ndvi",

    # GridMET
    "gridmet_rmax", "gridmet_rmin", "gridmet_sph",
    "gridmet_srad", "gridmet_tmmn", "gridmet_tmmx",
    "gridmet_vs",

    # Daymet
    "daymet_vp", "daymet_tmin", "daymet_tmax",
    "daymet_srad",
]

Load the full dataset.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path("CIMIS_WBGT_all_2000_2025_integrated_new.csv")

# All candidate predictors, excluding ERA5 WBGT
# because ERA5 WBGT is the baseline, not a model predictor.
all_features = [
    col for col in features
    if col != "era5_wbgt"
]

required_cols = list(dict.fromkeys(
    features + [
        "station_id",
        "year",
        "month",
        "date_local",
        "datetime_utc",
        "lat",
        "lon",
        "wbgt",
        "era5_wbgt"
    ]
))

chunks = []

for chunk in pd.read_csv(
    file_path,
    usecols=required_cols,
    chunksize=500_000,
    dtype={"station_id": "string"}
):
    chunk["year"] = pd.to_numeric(
        chunk["year"], errors="coerce"
    )

    chunk = chunk.loc[
        chunk["year"].between(2000, 2025)
    ]

    if not chunk.empty:
        chunks.append(chunk)

if not chunks:
    raise ValueError("No data found for years 2000–2025.")

df_full = pd.concat(chunks, ignore_index=True)

print("Total observations:", len(df_full))
print("Years:", sorted(df_full["year"].dropna().unique()))
print("Number of stations:", df_full["station_id"].nunique())

Prepare the target variable.

residual = CIMIS WBGT - ERA5 WBGT

In [ ]:
# Remove rows without target, baseline, or station ID
df_full = df_full.dropna(
    subset=["wbgt", "era5_wbgt", "station_id"]
).copy()

# Convert predictors to numeric
for col in all_features:
    df_full[col] = pd.to_numeric(
        df_full[col], errors="coerce"
    )

# Replace infinite values with NaN
df_full[all_features] = (
    df_full[all_features]
    .replace([np.inf, -np.inf], np.nan)
)

# Calculate residual target
df_full["wbgt_residual"] = (
    df_full["wbgt"] - df_full["era5_wbgt"]
)

df_full = df_full.reset_index(drop=True)

print("Observations after cleaning:", len(df_full))
print("Unique stations:", df_full["station_id"].nunique())
print("Years represented:", df_full["year"].nunique())

Create station-based train-test split (80/20).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df_full,
        groups=df_full["station_id"]
    )
)

train = df_full.iloc[train_idx].copy()
test = df_full.iloc[test_idx].copy()

print("Training observations:", len(train))
print("Testing observations:", len(test))

print("Training stations:", train["station_id"].nunique())
print("Testing stations:", test["station_id"].nunique())

# Verify no station overlap
train_stations = set(train["station_id"].unique())
test_stations = set(test["station_id"].unique())

print("Station overlap:", train_stations.intersection(test_stations))

Run 10-fold cross-validation LightGBM. 

It compares models with different predictor sets:
- All predictors
- Selected predictors with absolute correlation greater than 0.5
- Top predictors with absolute correlation greater than 0.4

In [ ]:
import time

from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Prepare CV data from the outer training stations
cv_data = train.copy()

y = cv_data["wbgt_residual"].to_numpy()
wbgt_values = cv_data["wbgt"].to_numpy()
era5_values = cv_data["era5_wbgt"].to_numpy()
groups = cv_data["station_id"].to_numpy()

print("Training observations:", len(cv_data))
print("Training stations:", cv_data["station_id"].nunique())

assert cv_data["station_id"].nunique() >= 10

# LightGBM parameters
lgbm_params = {
    "objective": "regression",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

# Create the same 10 folds for all predictor sets
cv = GroupKFold(n_splits=10)

fold_splits = list(
    cv.split(
        np.zeros(len(cv_data)),
        y,
        groups
    )
)

results = []

for fold, (train_idx, val_idx) in enumerate(
    fold_splits, start=1
):

    print(f"\n{'=' * 20} FOLD {fold} {'=' * 20}")

    fold_train = cv_data.iloc[train_idx]
    fold_val = cv_data.iloc[val_idx]

    # Calculate correlations using ONLY this fold's training data
    corr = (
        fold_train[features + ["wbgt"]]
        .corr(numeric_only=True)["wbgt"]
        .drop("wbgt")
        .sort_values(key=abs, ascending=False)
    )

    features_selected = corr[
        (corr.abs() > 0.5) &
        (corr.index != "era5_wbgt")
    ].index.tolist()

    features_top = corr[
        (corr.abs() > 0.4) &
        (corr.index != "era5_wbgt")
    ].index.tolist()

    # Predictor sets for this fold
    models = {
        "LightGBM - All predictors": all_features,
        "LightGBM - Selected predictors": features_selected,
        "LightGBM - Top predictors": features_top
    }

    print("Selected predictors:", len(features_selected))
    print("Top predictors:", len(features_top))

    # Targets and baseline for validation observations
    y_train = fold_train["wbgt_residual"].to_numpy()
    y_val = fold_val["wbgt_residual"].to_numpy()

    wbgt_val = fold_val["wbgt"].to_numpy()
    era5_val = fold_val["era5_wbgt"].to_numpy()

    for model_name, feature_cols in models.items():

        if len(feature_cols) == 0:
            print(f"Skipping {model_name}: no predictors selected.")
            continue

        X_train = (
            fold_train[feature_cols]
            .replace([np.inf, -np.inf], np.nan)
        )

        X_val = (
            fold_val[feature_cols]
            .replace([np.inf, -np.inf], np.nan)
        )

        model = LGBMRegressor(**lgbm_params)

        start_time = time.perf_counter()
        model.fit(X_train, y_train)
        training_time = time.perf_counter() - start_time

        # Predict residuals and reconstruct WBGT
        predicted_residual = model.predict(X_val)
        predicted_wbgt = era5_val + predicted_residual

        # Corrected WBGT metrics
        wbgt_rmse = np.sqrt(
            mean_squared_error(wbgt_val, predicted_wbgt)
        )
        wbgt_mae = mean_absolute_error(
            wbgt_val, predicted_wbgt
        )
        wbgt_bias = np.mean(predicted_wbgt - wbgt_val)
        wbgt_r2 = r2_score(wbgt_val, predicted_wbgt)

        # ERA5 baseline metrics
        era5_rmse = np.sqrt(
            mean_squared_error(wbgt_val, era5_val)
        )
        era5_mae = mean_absolute_error(wbgt_val, era5_val)
        era5_bias = np.mean(era5_val - wbgt_val)
        era5_r2 = r2_score(wbgt_val, era5_val)

        # Residual metrics
        residual_rmse = np.sqrt(
            mean_squared_error(y_val, predicted_residual)
        )
        residual_mae = mean_absolute_error(
            y_val, predicted_residual
        )
        residual_bias = np.mean(predicted_residual - y_val)
        residual_r2 = r2_score(y_val, predicted_residual)

        results.append({
            "Model": model_name,
            "Fold": fold,
            "Train_rows": len(train_idx),
            "Validation_rows": len(val_idx),
            "Train_stations": fold_train["station_id"].nunique(),
            "Validation_stations": fold_val["station_id"].nunique(),
            "Training_time_sec": training_time,
            "WBGT_RMSE": wbgt_rmse,
            "WBGT_MAE": wbgt_mae,
            "WBGT_Bias": wbgt_bias,
            "WBGT_R2": wbgt_r2,
            "ERA5_RMSE": era5_rmse,
            "ERA5_MAE": era5_mae,
            "ERA5_Bias": era5_bias,
            "ERA5_R2": era5_r2,
            "Residual_RMSE": residual_rmse,
            "Residual_MAE": residual_mae,
            "Residual_Bias": residual_bias,
            "Residual_R2": residual_r2
        })

        print(
            f"{model_name} | "
            f"RMSE: {wbgt_rmse:.4f} °C | "
            f"R²: {wbgt_r2:.4f} | "
            f"Time: {training_time:.2f} sec"
        )

results_df = pd.DataFrame(results)

display(results_df)

Summarize and save results.

In [ ]:
metrics = [
    "WBGT_RMSE",
    "WBGT_MAE",
    "WBGT_Bias",
    "WBGT_R2",
    "ERA5_RMSE",
    "ERA5_MAE",
    "ERA5_Bias",
    "ERA5_R2",
    "Residual_RMSE",
    "Residual_MAE",
    "Residual_Bias",
    "Residual_R2",
    "Training_time_sec"
]

summary = (
    results_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
)

display(summary.round(4))

results_df.to_csv(
    "lightgbm_2000_2025_cv_fold_results.csv",
    index=False
)

summary.to_csv(
    "lightgbm_2000_2025_cv_summary.csv"
)

After comparing the three predictor sets,
1. Choose the best-performing predictor set.
2. Train the final LightGBM model using all 80% training stations.
3. Predict residuals for the untouched 20% test stations.
4. Add predicted residuals to ERA5 WBGT.
5. Compare corrected WBGT against observed CIMIS WBGT and the original ERA5 baseline.